# Laboratori 04: Interpolimi, regresi dhe përshtatja e modeleve

**Lënda:** Simulim dhe Metoda Numerike  
**Kohëzgjatja:** 120 minuta  
**Mjedisi kryesor:** Python/NumPy; pjesa e dytë përmban zbatimin thelbësor në GNU Octave.

**Pyetja udhëheqëse:** Kur duhet një funksion të kalojë nëpër çdo pikë dhe kur duhet të përshkruajë prirjen e të dhënave?

Ky laborator nuk duhet lexuar si një listë komandash. Çdo pjesë nis nga modeli dhe nga një llogaritje që mund të kryhet me dorë; kodi përdoret më pas për të zgjeruar llogaritjen, për ta verifikuar dhe për të eksploruar regjime që nuk trajtohen dot me disa hapa algjebrikë. Ritmi i punës është

$$\text{pyetje}\longrightarrow\text{model}\longrightarrow\text{derivim}\longrightarrow\text{algoritëm}\longrightarrow\text{kod}\longrightarrow\text{kontroll}.$$


## Rezultatet e të nxënit

Në fund të seancës studenti duhet të jetë në gjendje të:

- dallojë interpolimin nga regresi;
- ndërtojë me dorë një polinom interpolues me tri pika;
- zbatojë formën e Njutonit dhe skemën baricentrike;
- shpjegojë dukurinë Runge dhe rolin e nyjeve të Çebishevit;
- nxjerrë kompleksitetin e ndërtimit dhe vlerësimit;
- përdorë mbetjet dhe numrin e kushtëzimit për diagnostikë;
- krahasojë ekuacionet normale me zgjidhjen QR/SVD të `lstsq`.


## Organizimi i seancës

| Koha | Veprimtaria | Produkti |
|---:|---|---|
| 0–15 min | Pyetja fizike dhe llogaritja e parë me dorë | Parashikim i shkruar |
| 15–40 min | Derivimi dhe algoritmi minimal | Formula dhe pseudokodi |
| 40–65 min | Zbatimi i udhëhequr në Python | Rezultat i verifikuar |
| 65–82 min | Kompleksiteti: formulë dhe matje | $T(n)$ dhe grafik kohor |
| 82–100 min | Qëndrueshmëria dhe diagnostika | Krahasim i dy rasteve |
| 100–112 min | Shembulli i plotë dhe punishte | Eksperiment i modifikuar |
| 112–120 min | GNU Octave dhe ekzekutimi lokal | Skript i riprodhueshëm |

Seksioni i shembullit të plotë është i detyrueshëm. Në punishte zgjidhen vetëm një ose dy ndryshime parametrash gjatë orës; pjesa tjetër mbetet për punë të pavarur.


## Si duhet lexuar ky laborator

Çdo metodë përshtatjeje fsheh një pohim për të dhënat. Interpolimi thotë se çdo vlerë duhet respektuar saktësisht; regresi thotë se vlerat përmbajnë luhatje dhe se kërkohet një prirje me numër më të vogël parametrash. Përpara zgjedhjes së algoritmit duhet pyetur se çfarë përfaqësojnë pikat: rezultate të një formule, matje me pacaktueshmëri apo mostra të një procesi stokastik.

Në këtë laborator stabiliteti lidhet fort me zgjedhjen e bazës dhe të nyjeve. Dy polinome matematikisht identike mund të llogariten me saktësi shumë të ndryshme në bazën monomiale, në formën e Njutonit ose në formën baricentrike. Prandaj nuk mjafton të thuhet “përdora polinom të gradës $n$”. Duhet të deklarohen baza, shkallëzimi dhe procedura e zgjidhjes.


## 1. Dy pyetje të ndryshme

Interpolimi kërkon një funksion $p$ të tillë që

$$p(x_i)=y_i,qquad i=0,\ldots,n.$$

Ai supozon se pikat janë vlera që duhen riprodhuar. Regresi pranon se matjet kanë zhurmë dhe kërkon parametra që minimizojnë një masë të mospërputhjes. Për modelin linear $y\approx a+bx$, metoda e katrorëve më të vegjël minimizon

$$S(a,b)=\sum_{i=1}^{m}[y_i-(a+bx_i)]^2.$$

Të detyrosh një polinom të kalojë nëpër çdo matje të zhurmshme zakonisht është mbipërshtatje, jo saktësi.


## 2. Interpolimi me dorë: tri pika

Për pikat $(0,1)$, $(1,2)$ dhe $(2,5)$, baza e Lagranzhit është

$$L_0(x)=\frac{(x-1)(x-2)}{(0-1)(0-2)}=\frac{(x-1)(x-2)}{2},$$

$$L_1(x)=\frac{x(x-2)}{(1-0)(1-2)}=-x(x-2),$$

$$L_2(x)=\frac{x(x-1)}{(2-0)(2-1)}=\frac{x(x-1)}{2}.$$

Prandaj

$$p(x)=1L_0(x)+2L_1(x)+5L_2(x)=x^2+1.$$

Kontrollet $p(0)=1$, $p(1)=2$, $p(2)=5$ janë pjesë e zgjidhjes. Para kodit llogaritni $p(1.5)=3.25$.


# Pjesa I — Python

## Përgatitja e mjedisit

Importet vendosen në një qelizë të vetme që eksperimenti të jetë i lexueshëm dhe i riprodhueshëm. Fara e gjeneratorit deklarohet edhe kur pjesa kryesore është deterministe; kështu çdo shtesë stokastike mund të përsëritet.


In [ ]:
import math
from time import perf_counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # realizim i riprodhueshëm
np.set_printoptions(precision=6, suppress=True)


In [ ]:
def lagrange_drejt(x_nyje, y_nyje, x):
    x_nyje=np.asarray(x_nyje,float); y_nyje=np.asarray(y_nyje,float)
    x=np.asarray(x,float); p=np.zeros_like(x)
    n=len(x_nyje)
    for i in range(n):
        Li=np.ones_like(x)
        for j in range(n):
            if i!=j:
                Li *= (x-x_nyje[j])/(x_nyje[i]-x_nyje[j])
        p += y_nyje[i]*Li
    return p

xn=np.array([0.,1.,2.]); yn=np.array([1.,2.,5.])
print('p(1.5)=',lagrange_drejt(xn,yn,1.5))
print('kontrolli në nyje:',lagrange_drejt(xn,yn,xn)-yn)


## 3. Forma e Njutonit dhe diferencat e ndara

Polinomi shkruhet

$$p_n(x)=a_0+a_1(x-x_0)+a_2(x-x_0)(x-x_1)+\cdots.$$

Koeficientët janë diferenca të ndara. Tabela trekëndore kërkon

$$1+2+\cdots+n=\frac{n(n+1)}{2}\in\Theta(n^2)$$

veprime, ndërsa vlerësimi i ndërthurur, analog me Horner-in, kërkon $\Theta(n)$. Kjo ndan qartë koston e parapërpunimit nga kostoja për çdo pikë të re.


In [ ]:
def koeficientet_njuton(x,y):
    x=np.asarray(x,float); a=np.asarray(y,float).copy(); n=len(x)
    for j in range(1,n):
        # Përditësimi nga fundi ruan vlerat që nevojiten ende.
        a[j:n]=(a[j:n]-a[j-1:n-1])/(x[j:n]-x[:n-j])
    return a

def vlereso_njuton(x_nyje,a,x):
    x=np.asarray(x,float); p=np.zeros_like(x)+a[-1]
    for k in range(len(a)-2,-1,-1):
        p=a[k]+(x-x_nyje[k])*p
    return p

a=koeficientet_njuton(xn,yn)
print('koeficientët:',a,'p(1.5)=',vlereso_njuton(xn,a,1.5))


## 4. Gabimi i interpolimit

Nëse $f$ ka derivatin e rendit $n+1$, ekziston $\xi$ brenda intervalit të nyjeve i tillë që

$$f(x)-p_n(x)=\frac{f^{(n+1)}(\xi)}{(n+1)!}\prod_{i=0}^{n}(x-x_i).$$

Gabimi varet jo vetëm nga lëmueshmëria e funksionit, por edhe nga vendosja e nyjeve. Rritja e gradës në nyje të baraslarguara nuk garanton përmirësim. Ky është mësimi i dukurisë Runge.


In [ ]:
def f_runge(x): return 1/(1+25*x*x)

xx=np.linspace(-1,1,1000)
fig,ax=plt.subplots(1,2,figsize=(11,4),sharey=True)
for a_plot,n in zip(ax,[11,21]):
    xe=np.linspace(-1,1,n); ye=f_runge(xe)
    ae=koeficientet_njuton(xe,ye)
    a_plot.plot(xx,f_runge(xx),'k',label='funksioni')
    a_plot.plot(xx,vlereso_njuton(xe,ae,xx),label=f'{n} nyje të baraslarguara')
    a_plot.plot(xe,ye,'o',ms=3); a_plot.set_ylim(-0.5,1.2); a_plot.grid(alpha=.3); a_plot.legend()
plt.show()


### 4.1 Nyjet e Çebishevit

Nyjet

$$x_k=\cos\left(\frac{(2k+1)\pi}{2(n+1)}\right),\qquad k=0,\ldots,n,$$

grumbullohen pranë skajeve dhe zvogëlojnë maksimumin e produktit $\prod_i|x-x_i|$. Ato nuk ndryshojnë funksionin; ndryshojnë projektimin e eksperimentit numerik.


In [ ]:
n=21
k=np.arange(n)
xc=np.cos((2*k+1)*np.pi/(2*n)); yc=f_runge(xc)
ac=koeficientet_njuton(xc,yc)
xe=np.linspace(-1,1,n); ae=koeficientet_njuton(xe,f_runge(xe))
err_e=np.max(np.abs(vlereso_njuton(xe,ae,xx)-f_runge(xx)))
err_c=np.max(np.abs(vlereso_njuton(xc,ac,xx)-f_runge(xx)))
print(f'gabimi maksimal: baraslarguara={err_e:.3e}, Çebishev={err_c:.3e}')


## Si do ta llogarisim kompleksitetin

Kompleksiteti nuk përcaktohet fillimisht me kronometër. Së pari zgjedhim madhësinë e hyrjes $n$, identifikojmë veprimin dominues dhe nxjerrim formulën $T(n)$. Vetëm pasi kemi një parashikim matematikor kryejmë kohëmatjen.

Për një cikël që kryen $c$ veprime në secilin prej $n$ hapave,

$$T(n)=cn+d\in\Theta(n).$$

Për dy cikle të plota të ndërfutura,

$$T(n)=\sum_{i=1}^{n}\sum_{j=1}^{n}c=cn^2\in\Theta(n^2).$$

Koha e matur modelohet si $t(n)\approx Cn^p$. Në koordinata log–log,

$$\log t=\log C+p\log n,$$

prandaj pjerrësia $p$ krahasohet me rendin e parashikuar. Matja mund të ndikohet nga kostot fikse, interpretuesi dhe memoria; ajo është kontroll empirik, jo zëvendësim i derivimit.


### 5.1 Numërimi i veprimeve

Për formulën e drejtpërdrejtë të Lagranzhit, për secilin prej $n+1$ terma ndërtohet një produkt me $n$ faktorë. Një vlerësim kushton

$$T_L(n)\approx(n+1)n+(n+1)\in\Theta(n^2).$$

Forma e Njutonit kushton $\Theta(n^2)$ një herë për koeficientët dhe $\Theta(n)$ për çdo vlerësim. Për $M$ pika të reja, kostoja është

$$T_N(n,M)=\Theta(n^2+Mn).$$

Kjo formulë shpjegon pse parapërpunimi është i dobishëm kur polinomi vlerësohet shumë herë.


In [ ]:
def kohematje(f,*args,perseritje=3):
    ts=[]
    for _ in range(perseritje):
        t0=perf_counter(); f(*args); ts.append(perf_counter()-t0)
    return min(ts)

ns=np.array([10,20,40,80,160])
t_l=[]; t_n=[]
for n in ns:
    x=np.linspace(-1,1,n); y=np.sin(x); z=np.linspace(-.9,.9,200)
    t_l.append(kohematje(lagrange_drejt,x,y,z))
    a=koeficientet_njuton(x,y)
    t_n.append(kohematje(vlereso_njuton,x,a,z))
print('pjerrësi Lagranzh=',np.polyfit(np.log(ns),np.log(t_l),1)[0])
print('pjerrësi Njuton  =',np.polyfit(np.log(ns),np.log(t_n),1)[0])


## Si do ta analizojmë qëndrueshmërinë

**Kushtëzimi** i përket problemit: sa ndryshon zgjidhja e saktë kur të dhënat perturbohen pak? **Qëndrueshmëria** i përket algoritmit: sa gabim shton procedura e llogaritjes? Në trajtë skematike,

$$\text{gabimi përpara}\lesssim
\text{numri i kushtëzimit}\times\text{gabimi prapa}.$$

Në secilin laborator do të përdorim një provë konkrete: mbetje, ligj ruajtjeje, kufi stabiliteti, rezultat analitik ose përsëritje me hap më të vogël. Fjalia “grafiku duket mirë” nuk është provë numerike.


### 6.1 Matrica Vandermonde dhe shkallëzimi

Interpolimi ose regresi polinomial mund të shkruhet me matricën $V_{ij}=x_i^j$. Fuqitë e larta dhe intervalet e mëdha prodhojnë kolona me shkallë shumë të ndryshme, duke rritur $\kappa(V)$. Zhvendosja dhe shkallëzimi

$$z=\frac{x-\bar{x}}{s_x}$$

e bëjnë problemin më të kontrollueshëm. Për regresin, formimi i $X^TX$ afërsisht katrorëzon numrin e kushtëzimit:

$$\kappa(X^TX)\approx\kappa(X)^2.$$

Prandaj `np.linalg.lstsq`, e mbështetur në QR/SVD, preferohet ndaj llogaritjes së $(X^TX)^{-1}X^T\mathbf y$.


In [ ]:
x=np.linspace(1000,1010,30)
V=np.vander(x,8,increasing=True)
z=(x-x.mean())/x.std()
Vz=np.vander(z,8,increasing=True)
print('kappa pa shkallëzim =',np.linalg.cond(V))
print('kappa me shkallëzim =',np.linalg.cond(Vz))


## 7. Regresi linear i nxjerrë me dorë

Nga $\partial S/\partial a=0$ dhe $\partial S/\partial b=0$ merren ekuacionet normale

$$ma+b\sum x_i=\sum y_i,$$

$$a\sum x_i+b\sum x_i^2=\sum x_iy_i.$$

Pas qendërzimit, zgjidhja bëhet

$$b=\frac{\sum(x_i-\bar x)(y_i-\bar y)}{\sum(x_i-\bar x)^2},\qquad a=\bar y-b\bar x.$$

Kjo formulë është e thjeshtë për dy parametra. Për modele më të mëdha përdorim formulimin matricor dhe algoritme të qëndrueshme.


In [ ]:
x=np.linspace(0,10,40)
y=2.0+0.7*x+rng.normal(0,0.8,size=x.size)
b=np.sum((x-x.mean())*(y-y.mean()))/np.sum((x-x.mean())**2)
a=y.mean()-b*x.mean()
X=np.column_stack([np.ones_like(x),x])
beta,_,_,_=np.linalg.lstsq(X,y,rcond=None)
mbetje=y-X@beta
print('me dorë:',a,b,'lstsq:',beta,'RMSE=',np.sqrt(np.mean(mbetje**2)))


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].scatter(x,y,s=20,label='të dhëna'); ax[0].plot(x,X@beta,'r',label='regresi')
ax[0].set(xlabel='x',ylabel='y'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].axhline(0,color='k',lw=1); ax[1].scatter(x,mbetje,s=20)
ax[1].set(xlabel='x',ylabel='mbetja'); ax[1].grid(alpha=.3)
plt.show()


## 8. Shembulli i plotë dhe punishte

Në shembullin e plotë krahasoni: polinom interpolues me të gjitha pikat, regres polinomial me gradë të ulët dhe regres me gradë të lartë. Ndryshoni vetëm një nga: numri i pikave, niveli i zhurmës, grada ose vendosja e nyjeve. Për çdo rast raportoni gabimin në të dhënat e trajnimit dhe në një rrjetë të dendur. Një gabim i vogël në pikat e dhëna nuk garanton sjellje të mirë midis tyre.


In [ ]:
def punishte_regresi(grada=5,zhurma=0.15,n=25):
    x=np.linspace(-1,1,n); y=f_runge(x)+rng.normal(0,zhurma,n)
    z=(x-x.mean())/x.std(); Z=np.vander(z,grada+1,increasing=True)
    beta=np.linalg.lstsq(Z,y,rcond=None)[0]
    xx=np.linspace(-1,1,500); zz=(xx-x.mean())/x.std()
    yy=np.vander(zz,grada+1,increasing=True)@beta
    rmse=np.sqrt(np.mean((np.vander(z,grada+1,increasing=True)@beta-y)**2))
    return x,y,xx,yy,rmse,np.linalg.cond(Z)

x,y,xx,yy,rmse,kappa=punishte_regresi(7)
print('RMSE=',rmse,'kappa=',kappa)
plt.plot(xx,f_runge(xx),'k--',label='modeli pa zhurmë'); plt.plot(xx,yy,label='regresi')
plt.scatter(x,y,s=20); plt.grid(alpha=.3); plt.legend(); plt.show()


### Interpolimi nuk autorizon ekstrapolim të pakufizuar

Formula e mbetjes përmban produktin $\prod_i(x-x_i)$. Jashtë intervalit të nyjeve ky produkt mund të rritet shpejt. Edhe një përshtatje shumë e mirë brenda intervalit nuk justifikon automatikisht parashikime larg të dhënave. Në raport duhet shënuar qartë intervali i interpolimit dhe çdo pikë ekstrapolimi.


In [ ]:
# Shembull i rrezikut të ekstrapolimit me të dhëna pa zhurmë.
xtr=np.linspace(0,1,8); ytr=np.exp(xtr); atr=koeficientet_njuton(xtr,ytr)
for xp in [0.5,1.0,1.5,2.0]:
    yp=vlereso_njuton(xtr,atr,xp)
    print(f'x={xp}: p={yp:.8f}, exp={np.exp(xp):.8f}, gabimi={abs(yp-np.exp(xp)):.2e}')


### Ndarja trajnim–kontroll në një shembull numerik

Rritja e gradës e ul gjithmonë ose pothuajse gjithmonë gabimin në të dhënat e përdorura për përshtatje. Kriteri më i dobishëm është gabimi në pika që nuk u përdorën për përcaktimin e koeficientëve. Edhe pa hyrë ende në mësimin makinerik, kjo ndarje tregon mbipërshtatjen dhe ndihmon në zgjedhjen e gradës.


In [ ]:
xall=np.linspace(-1,1,60); yall=f_runge(xall)+rng.normal(0,.08,len(xall))
train=np.arange(len(xall))%3!=0; test=~train
for deg in [1,3,5,9,15]:
    mu=xall[train].mean(); sd=xall[train].std()
    Xtr=np.vander((xall[train]-mu)/sd,deg+1,increasing=True)
    beta=np.linalg.lstsq(Xtr,yall[train],rcond=None)[0]
    pred_tr=Xtr@beta
    pred_te=np.vander((xall[test]-mu)/sd,deg+1,increasing=True)@beta
    print(deg,np.sqrt(np.mean((pred_tr-yall[train])**2)),np.sqrt(np.mean((pred_te-yall[test])**2)))


### Interpretimi i mbetjeve

Mbetjet duhet të luhaten rreth zeros pa strukturë të dukshme. Një hark sistematik sugjeron se modeli është tepër i thjeshtë; një gjerësi që rritet me $x$ sugjeron variancë jo të njëtrajtshme; një pikë shumë e largët mund të ketë ndikim të madh. Këto vëzhgime janë pjesë e analizës së modelit, jo zbukurime grafike.


In [ ]:
# Kontroll ortogonaliteti: në LS, X^T r duhet të jetë afër zeros.
ortho=X.T@mbetje
print('X^T r =',ortho,'norma=',np.linalg.norm(ortho))
assert np.linalg.norm(ortho)<1e-10


### Pyetje për vetëkontroll

1. Kur është gabim konceptual të përdoret interpolimi?
2. Pse nyjet e Çebishevit ndihmojnë pranë skajeve?
3. Si ndahet kostoja e parapërpunimit nga ajo e vlerësimit?
4. Pse zgjidhja me `lstsq` preferohet ndaj formimit të anasjelltës së $X^TX$?


# Pjesa II — GNU Octave

## 9. Interpolimi dhe regresi

```octave
x=[0,1,2]; y=[1,2,5];
p=polyfit(x,y,2);                 % koeficientët në bazën e fuqive
fprintf('p(1.5)=%.12g\n',polyval(p,1.5));

x=linspace(0,10,40)';
randn('seed',2026); y=2+0.7*x+0.8*randn(size(x));
X=[ones(size(x)),x];
beta=X\y;                          % QR, jo inv(X'*X)*X'*y
mbetje=y-X*beta;
fprintf('a=%.6f b=%.6f RMSE=%.6f\n',beta(1),beta(2),sqrt(mean(mbetje.^2)));
```

## 10. Nyjet e Çebishevit

```octave
f=@(x) 1./(1+25*x.^2); n=21; xx=linspace(-1,1,1000);
xe=linspace(-1,1,n); pe=polyfit(xe,f(xe),n-1);
k=0:n-1; xc=cos((2*k+1)*pi/(2*n)); pc=polyfit(xc,f(xc),n-1);
fprintf('gabimi baraslarguar %.3e\n',max(abs(polyval(pe,xx)-f(xx))));
fprintf('gabimi Chebyshev    %.3e\n',max(abs(polyval(pc,xx)-f(xx))));
```

Për grada të larta, `polyfit` në bazën monomiale mund të jetë vetë i keqkushtëzuar; eksperimenti duhet interpretuar bashkë me paralajmërimet dhe numrin e kushtëzimit.


## Ekzekutimi jashtë notebook-ut

Një notebook e ndihmon shpjegimin, por studenti duhet të dijë të ekzekutojë edhe një program të zakonshëm. Qeliza vijuese krijon një skript minimal në `scripts_local/`. Ai përmban eksperimentin qendror dhe mund të ndryshohet me një editor teksti.


In [ ]:
script_dir = Path('scripts_local')
script_dir.mkdir(exist_ok=True)
script_text = "import numpy as np\nx=np.linspace(0,10,40); rng=np.random.default_rng(2026)\ny=2+0.7*x+rng.normal(0,0.8,x.size)\nX=np.c_[np.ones_like(x),x]\nb=np.linalg.lstsq(X,y,rcond=None)[0]\nprint('beta=',b,'RMSE=',np.sqrt(np.mean((y-X@b)**2)))\n"
script_path = script_dir / 'laboratori_04_interpolimi_regresi.py'
script_path.write_text(script_text, encoding='utf-8')
print('U krijua:', script_path)


Ekzekutimi në terminal:

```bash
python3 scripts_local/laboratori_04_interpolimi_regresi.py
```

Mos raportoni vetëm se programi u ekzekutua. Shënoni komandën, një dalje numerike, një kontroll dhe një përfundim.

## Raporti i shkurtër dhe kontrolli përfundimtar

Raporti duhet të përmbajë pyetjen, modelin, supozimet, derivimin kryesor, analizën e kompleksitetit, provën e qëndrueshmërisë, figurën me boshte e njësi dhe interpretimin fizik.

- [ ] Ndava interpolimin nga regresi.
- [ ] Nxora kostot e ndërtimit dhe vlerësimit.
- [ ] Kontrollova mbetjet dhe numrin e kushtëzimit.
- [ ] Krahasova nyjet e baraslarguara me nyjet e Çebishevit.
- [ ] Ekzekutova skriptin Python nga terminali.
- [ ] Identifikova fragmentin përkatës në GNU Octave.
